# β-VAE Training — Credit Card Fraud Detection

## 1. Clone Repository & Install Dependencies

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/nanokwok/Deep-Fraud-VAE.git'
REPO_DIR = '/content/Deep-Fraud-VAE'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('Repo already cloned — pulling latest')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — consider enabling Runtime > Change runtime type > GPU')

## 2. Download Dataset from Kaggle

In [ ]:
import os, json

# TODO: Replace with your Kaggle credentials. Do NOT commit.
KAGGLE_USERNAME = 'your_kaggle_username'
KAGGLE_KEY      = 'your_kaggle_api_key'

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(f'{kaggle_dir}/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
print('Kaggle credentials saved.')

In [ ]:
import os, subprocess, sys

RAW_DIR = 'data/raw'
RAW_CSV = f'{RAW_DIR}/creditcard.csv'
os.makedirs(RAW_DIR, exist_ok=True)

if os.path.exists(RAW_CSV):
    size_mb = os.path.getsize(RAW_CSV) / 1e6
    print(f'creditcard.csv already present ({size_mb:.1f} MB) — skipping download.')
else:
    print('Downloading Credit Card Fraud dataset from Kaggle (~144 MB)...')
    # Dataset: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
    subprocess.run([
        sys.executable, '-m', 'kaggle', 'datasets', 'download',
        'mlg-ulb/creditcardfraud',
        '-p', RAW_DIR, '--unzip',
    ], check=True)
    size_mb = os.path.getsize(RAW_CSV) / 1e6
    print(f'Download complete: {size_mb:.1f} MB')

## 3. Run Preprocessing Pipeline

Runs `src/preprocess.py` to generate the `.npy` feature arrays.

Steps: load `creditcard.csv` → semi-supervised split (80% normal → train, 20% normal + 100% fraud → val) → `StandardScaler` on Time, `RobustScaler` on Amount, V1–V28 untouched → save.

Expected runtime: < 30 seconds.

In [ ]:
import subprocess, sys, numpy as np, json
from pathlib import Path

PROC_DIR = Path('data/processed')
REQUIRED = ['X_train.npy', 'X_val.npy', 'y_train.npy', 'y_val.npy',
            'feature_columns.json', 'scaler.pkl']

if all((PROC_DIR / f).exists() for f in REQUIRED):
    print('Processed arrays already exist — skipping. Delete data/processed/ to rerun.')
else:
    print('Running preprocessing...')
    subprocess.run([sys.executable, '-m', 'src.preprocess'], check=True)

X_train = np.load(PROC_DIR / 'X_train.npy')
X_val   = np.load(PROC_DIR / 'X_val.npy')
y_val   = np.load(PROC_DIR / 'y_val.npy')
with open(PROC_DIR / 'feature_columns.json') as f:
    feat_names = json.load(f)

print(f'X_train : {X_train.shape}  dtype={X_train.dtype}  (normal only)')
print(f'X_val   : {X_val.shape}  dtype={X_val.dtype}')
print(f'y_val   : fraud_rate={y_val.mean()*100:.4f}%  fraud_count={y_val.sum()}')
print(f'features: {feat_names}')
assert not np.isnan(X_train).any(), 'NaN detected in X_train!'
assert not np.isnan(X_val).any(),   'NaN detected in X_val!'
print('NaN check passed.')

## 4. Verify GPU & Model Architecture

In [ ]:
import torch, numpy as np
from pathlib import Path
import sys; sys.path.insert(0, '.')

device = (
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else 'cpu'
)
print(f'Device     : {device}')
if device == 'cuda':
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
elif device == 'cpu':
    print('WARNING: CPU only — training will be slower.')

proc = Path('data/processed')
X_train = np.load(proc / 'X_train.npy')
y_val   = np.load(proc / 'y_val.npy')
print(f'train rows : {len(X_train):,}  (normal transactions only)')
print(f'val normal : {(y_val==0).sum():,}  |  val fraud: {(y_val==1).sum():,}')

from src.model import BetaVAE
from src.config import BETA, LATENT_DIM, ENCODER_DIMS, DECODER_DIMS, INPUT_DIM
model    = BetaVAE(input_dim=INPUT_DIM)
n_params = sum(p.numel() for p in model.parameters())
print(f'\nBetaVAE architecture:')
print(f'  Encoder : {INPUT_DIM} → {" → ".join(str(d) for d in ENCODER_DIMS)} → μ/logσ² ({LATENT_DIM})')
print(f'  Decoder : {LATENT_DIM} → {" → ".join(str(d) for d in DECODER_DIMS)} → {INPUT_DIM}')
print(f'  β={BETA}  latent_dim={LATENT_DIM}  total_params={n_params:,}')

## 5. Train the VAE

- Trains on **normal transactions only** (Class 0).
- Validation uses the full val set (normal + fraud) to compute anomaly scores.
- **Early stopping on Val AUPRC** (patience = 10) — AUPRC is the correct metric for 0.17% fraud imbalance.
- Best checkpoint saved to `models/best_vae.pth` only when AUPRC improves.
- Uncomment the `cfg.EXPERIMENTS_DIR` line to persist to Google Drive.

In [ ]:
import sys, importlib
if '.' not in sys.path:
    sys.path.insert(0, '.')

import src.config as cfg
# Uncomment to persist experiments across Colab VM resets:
# cfg.EXPERIMENTS_DIR = '/content/drive/MyDrive/Fraud_VAE_Project'

# Reload src modules so any config changes take effect
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'):
        del sys.modules[mod]

from src.train import train
exp_dir = train()   # returns Path to this run's experiment folder

print()
print('Experiment artefacts saved to:', exp_dir)
import os
for fname in sorted(os.listdir(str(exp_dir))):
    size = os.path.getsize(os.path.join(str(exp_dir), fname))
    print(f'  {fname:<30} {size/1024:.1f} KB')

## 6. Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_csv = exp_dir / 'loss_history.csv'
log_df  = pd.read_csv(log_csv)
print(f'Epochs trained : {len(log_df)}')
print(log_df[['epoch', 'train_loss', 'val_loss', 'val_auroc', 'val_auprc']].tail(5).to_string(index=False))

best_ep = int(log_df.loc[log_df['val_auprc'].idxmax(), 'epoch'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
ax = axes[0]
ax.plot(log_df['epoch'], log_df['train_loss'], label='Train Loss', color='steelblue')
ax.plot(log_df['epoch'], log_df['val_loss'],   label='Val Loss',   color='crimson', linestyle='--')
ax.axvline(best_ep, color='orange', linewidth=1.2, linestyle=':', label=f'best ep {best_ep}')
ax.set_xlabel('Epoch'); ax.set_title('β-ELBO Loss'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# AUROC
ax = axes[1]
ax.plot(log_df['epoch'], log_df['val_auroc'], color='steelblue', label='Val AUROC')
ax.axvline(best_ep, color='orange', linewidth=1.2, linestyle=':')
ax.set_xlabel('Epoch'); ax.set_title('Val AUROC'); ax.set_ylim(0, 1)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# AUPRC
ax = axes[2]
ax.plot(log_df['epoch'], log_df['val_auprc'], color='crimson', label='Val AUPRC')
ax.axvline(best_ep, color='orange', linewidth=1.2, linestyle=':', label=f'best ep {best_ep}')
ax.set_xlabel('Epoch'); ax.set_title('Val AUPRC  (early-stopping metric)'); ax.set_ylim(0, 1)
ax.legend(fontsize=8); ax.grid(alpha=0.3)

best_auprc = log_df['val_auprc'].max()
best_auroc = log_df.loc[log_df['val_auprc'].idxmax(), 'val_auroc']
plt.suptitle(
    f'β-VAE Training  (best epoch={best_ep}  AUPRC={best_auprc:.4f}  AUROC={best_auroc:.4f})',
    fontsize=12,
)
plt.tight_layout()
import os; os.makedirs('reports/figures', exist_ok=True)
plt.savefig('reports/figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best epoch: {best_ep}  |  AUPRC={best_auprc:.4f}  |  AUROC={best_auroc:.4f}')

## 7. Anomaly Score Distribution

Reconstruction error (mean squared error per sample) is the anomaly score.

**Expected:** Fraud histogram shifted right of Normal.  
**If heavily overlapping:** lower `BETA` in `src/config.py` and retrain.

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, roc_auc_score
from pathlib import Path
from src.model import BetaVAE
from src.config import DATA_PROC_DIR, INPUT_DIM

proc   = Path(DATA_PROC_DIR)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt  = torch.load('models/best_vae.pth', map_location=device)
model = BetaVAE(input_dim=INPUT_DIM).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Checkpoint: epoch={ckpt['epoch']}  AUPRC={ckpt['val_auprc']:.4f}  AUROC={ckpt['val_auroc']:.4f}")

X_val = torch.from_numpy(np.load(proc / 'X_val.npy').astype('float32')).to(device)
y_val = np.load(proc / 'y_val.npy')

with torch.no_grad():
    x_hat, _, _ = model(X_val)
    scores = ((X_val - x_hat) ** 2).mean(dim=1).cpu().numpy()

normal_s = scores[y_val == 0]
fraud_s  = scores[y_val == 1]
sep      = fraud_s.mean() / (normal_s.mean() + 1e-9)

auroc = roc_auc_score(y_val, scores)
auprc = average_precision_score(y_val, scores)
print(f'Normal mean={normal_s.mean():.4f}  p95={np.percentile(normal_s, 95):.4f}')
print(f'Fraud  mean={fraud_s.mean():.4f}  p95={np.percentile(fraud_s,  95):.4f}')
print(f'Separation : {sep:.2f}×')
print(f'AUROC      : {auroc:.4f}')
print(f'AUPRC      : {auprc:.4f}')

clip = float(np.percentile(scores, 99))
bins = np.linspace(0, clip, 80)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(normal_s.clip(max=clip), bins=bins, alpha=0.6, density=True,
        color='steelblue', label=f'Normal  (n={len(normal_s):,})')
ax.hist(fraud_s.clip(max=clip),  bins=bins, alpha=0.6, density=True,
        color='crimson',   label=f'Fraud   (n={len(fraud_s):,})')
ax.set_xlabel('Reconstruction Error (anomaly score)')
ax.set_ylabel('Density')
ax.set_title(
    f'Anomaly Score Distribution — Val Set\n'
    f'Separation={sep:.2f}×  |  AUROC={auroc:.4f}  |  AUPRC={auprc:.4f}'
)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('reports/figures/anomaly_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()